In [9]:
from pathlib import Path
import numpy as np


BASE_DIR = Path("..")

In [11]:
# ============================================================
# CELL 24
# Load Saved Unlabeled Text Embeddings
# ============================================================

import numpy as np


print("=" * 90)
print("LOAD UNLABELED TEXT EMBEDDINGS")
print("=" * 90)


UNLABELED_TEXT_EMBED_PATH = (
    BASE_DIR
    / "final_data"
    / "embeddings"
    / "xlmr"
    / "unlabeled_description_embeddings.npy"
)


unlabeled_text_embeddings = np.load(
    UNLABELED_TEXT_EMBED_PATH
)


print(
    "Loaded:",
    UNLABELED_TEXT_EMBED_PATH
)

print(
    "Shape:",
    unlabeled_text_embeddings.shape
)

print(
    "Dtype:",
    unlabeled_text_embeddings.dtype
)


assert (
    len(unlabeled_text_embeddings)
    == len(true_unlabeled)
), "Text embeddings and unlabeled users are not aligned."

LOAD UNLABELED TEXT EMBEDDINGS
Loaded: ..\final_data\embeddings\xlmr\unlabeled_description_embeddings.npy
Shape: (18331, 768)
Dtype: float32


In [12]:
print("="*90)
print("19K MODALITY AVAILABILITY")
print("="*90)


availability_cols = [
    "has_profile",
    "has_description",
    "has_raw_tweets",
    "has_behavior_temporal",
    "has_graph"
]


for col in availability_cols:
    
    print(
        f"{col}:",
        true_unlabeled[col].sum(),
        "/",
        len(true_unlabeled),
        f"({true_unlabeled[col].mean()*100:.2f}%)"
    )

19K MODALITY AVAILABILITY
has_profile: 18331 / 18331 (100.00%)
has_description: 11647 / 18331 (63.54%)
has_raw_tweets: 0 / 18331 (0.00%)
has_behavior_temporal: 18331 / 18331 (100.00%)
has_graph: 3033 / 18331 (16.55%)


In [13]:
# ============================================================
# GRAPH COVERAGE CHECK
# ============================================================

from pathlib import Path
import pickle
import pandas as pd
import numpy as np


BASE_DIR = Path("..")


print("=" * 90)
print("GRAPH COVERAGE CHECK")
print("=" * 90)


# ------------------------------------------------------------
# Load true unlabeled users if not already loaded
# ------------------------------------------------------------

if "true_unlabeled" not in globals():

    TRUE_UNLABELED_PATH = (
        BASE_DIR
        / "final_data"
        / "users"
        / "modeling_true_unlabeled.csv"
    )

    true_unlabeled = pd.read_csv(
        TRUE_UNLABELED_PATH
    )

    print(
        "Loaded true_unlabeled:",
        true_unlabeled.shape
    )


# ------------------------------------------------------------
# Load graph node mapping
# ------------------------------------------------------------

GRAPH_DIR = (
    BASE_DIR
    / "final_data"
    / "graph"
)


with open(
    GRAPH_DIR / "node_to_id.pkl",
    "rb"
) as f:

    node_to_id = pickle.load(f)


print(
    "Graph nodes:",
    len(node_to_id)
)


# ------------------------------------------------------------
# Match graph nodes with unlabeled users
# ------------------------------------------------------------

user_keys = (
    true_unlabeled["user_key"]
    .astype(str)
    .str.strip()
    .str.lower()
)


graph_keys = {
    str(k).strip().lower()
    for k in node_to_id.keys()
}


matched = user_keys.isin(
    graph_keys
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    "Unlabeled users:",
    len(true_unlabeled)
)

print(
    "Matched users:",
    matched.sum()
)

print(
    "Unmatched users:",
    (~matched).sum()
)

print(
    "Coverage:",
    f"{matched.mean() * 100:.2f}%"
)

GRAPH COVERAGE CHECK
Graph nodes: 13466
Unlabeled users: 18331
Matched users: 12397
Unmatched users: 5934
Coverage: 67.63%


In [14]:
# ============================================================
# CORRECTED GRAPH COVERAGE CONSISTENCY CHECK
# ============================================================

import numpy as np
import pandas as pd


print("=" * 90)
print("CORRECTED GRAPH COVERAGE CONSISTENCY CHECK")
print("=" * 90)


# ------------------------------------------------------------
# Normalize user keys
# ------------------------------------------------------------

user_keys = (
    true_unlabeled["user_key"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# Normalize node_to_id mapping
# IMPORTANT: keep both normalized key AND node id
# ------------------------------------------------------------

normalized_node_to_id = {
    str(key).strip().lower(): node_id
    for key, node_id in node_to_id.items()
}


# ------------------------------------------------------------
# Original has_graph flag
# ------------------------------------------------------------

has_graph_flag = (
    true_unlabeled["has_graph"]
    .fillna(0)
    .astype(int)
    .eq(1)
)


# ------------------------------------------------------------
# Membership in graph
# ------------------------------------------------------------

graph_member = user_keys.isin(
    normalized_node_to_id
)


print("\nOriginal has_graph:")
print(
    has_graph_flag.value_counts()
)


print("\nIn node_to_id:")
print(
    graph_member.value_counts()
)


print("\nComparison:")

print(
    pd.crosstab(
        has_graph_flag,
        graph_member,
        rownames=["has_graph"],
        colnames=["in_node_to_id"]
    )
)


# ------------------------------------------------------------
# Load actual degree features
# ------------------------------------------------------------

degree_features = np.load(
    GRAPH_DIR / "degree_features.npy"
)


print("\nDegree feature matrix:")
print(
    "Shape:",
    degree_features.shape
)


# ------------------------------------------------------------
# Verify actual usable graph features
# ------------------------------------------------------------

usable_graph = []

for key in user_keys:

    node_id = normalized_node_to_id.get(key)

    usable = (
        node_id is not None
        and 0 <= node_id < len(degree_features)
    )

    usable_graph.append(usable)


usable_graph = np.asarray(
    usable_graph,
    dtype=bool
)


print("\nActually usable graph features:")

print(
    "Users:",
    usable_graph.sum()
)

print(
    "Missing:",
    (~usable_graph).sum()
)

print(
    "Coverage:",
    f"{usable_graph.mean() * 100:.2f}%"
)

CORRECTED GRAPH COVERAGE CONSISTENCY CHECK

Original has_graph:
has_graph
False    15298
True      3033
Name: count, dtype: int64

In node_to_id:
user_key
True     12397
False     5934
Name: count, dtype: int64

Comparison:
in_node_to_id  False  True 
has_graph                  
False           5886   9412
True              48   2985

Degree feature matrix:
Shape: (13466, 3)

Actually usable graph features:
Users: 12397
Missing: 5934
Coverage: 67.63%


In [15]:
# ============================================================
# GRAPH FEATURE ALIGNMENT CHECK
# ============================================================

import numpy as np
import pickle


print("=" * 90)
print("GRAPH FEATURE ALIGNMENT CHECK")
print("=" * 90)


GRAPH_DIR = (
    BASE_DIR
    / "final_data"
    / "graph"
)


# ------------------------------------------------------------
# Load node mapping
# ------------------------------------------------------------

with open(
    GRAPH_DIR / "node_to_id.pkl",
    "rb"
) as f:
    node_to_id = pickle.load(f)


# Normalize mapping keys
normalized_node_to_id = {
    str(key).strip().lower(): node_id
    for key, node_id in node_to_id.items()
}


# ------------------------------------------------------------
# Load degree features
# ------------------------------------------------------------

degree_features = np.load(
    GRAPH_DIR / "degree_features.npy"
)


print(
    "Nodes in mapping:",
    len(normalized_node_to_id)
)

print(
    "Degree feature shape:",
    degree_features.shape
)

print(
    "Feature dimension:",
    degree_features.shape[1]
)


# ------------------------------------------------------------
# Check matched node ids
# ------------------------------------------------------------

matched_nodes = []


user_keys = (
    true_unlabeled["user_key"]
    .astype(str)
    .str.strip()
    .str.lower()
)


for key in user_keys:

    node_id = normalized_node_to_id.get(key)

    if node_id is not None:

        matched_nodes.append(
            node_id
        )


matched_nodes = np.asarray(
    matched_nodes,
    dtype=int
)


print(
    "\nMatched node ids:",
    len(matched_nodes)
)


if len(matched_nodes) > 0:

    print(
        "Max node id:",
        matched_nodes.max()
    )

    print(
        "Min node id:",
        matched_nodes.min()
    )

    print(
        "Valid node ids:",
        np.all(
            (matched_nodes >= 0)
            &
            (matched_nodes < len(degree_features))
        )
    )

GRAPH FEATURE ALIGNMENT CHECK
Nodes in mapping: 13465
Degree feature shape: (13466, 3)
Feature dimension: 3

Matched node ids: 12397
Max node id: 13465
Min node id: 0
Valid node ids: True


In [16]:
# ============================================================
# GRAPH KEY COLLISION CHECK
# ============================================================

from collections import defaultdict


print("=" * 90)
print("GRAPH KEY COLLISION CHECK")
print("=" * 90)


normalized_groups = defaultdict(list)


for original_key, node_id in node_to_id.items():

    normalized_key = (
        str(original_key)
        .strip()
        .lower()
    )

    normalized_groups[normalized_key].append(
        (original_key, node_id)
    )


collisions = {
    key: values
    for key, values in normalized_groups.items()
    if len(values) > 1
}


print(
    "Original mapping size:",
    len(node_to_id)
)

print(
    "Normalized unique keys:",
    len(normalized_groups)
)

print(
    "Collision groups:",
    len(collisions)
)


for normalized_key, values in collisions.items():

    print("\nNormalized key:", normalized_key)

    for original_key, node_id in values:

        print(
            "Original key:",
            repr(original_key),
            "| Node ID:",
            node_id,
            "| Features:",
            degree_features[node_id]
        )

GRAPH KEY COLLISION CHECK
Original mapping size: 13466
Normalized unique keys: 13465
Collision groups: 1

Normalized key: mehdimmj
Original key: 'MehdiMMJ' | Node ID: 10229 | Features: [ 64.  64. 128.]
Original key: 'mehdimmj' | Node ID: 10778 | Features: [ 73.  73. 146.]


In [17]:
# ============================================================
# SAFE GRAPH NODE RESOLUTION
# ============================================================

from collections import defaultdict
import numpy as np


print("=" * 90)
print("SAFE GRAPH NODE RESOLUTION")
print("=" * 90)


# ------------------------------------------------------------
# Build normalized groups without overwriting collisions
# ------------------------------------------------------------

normalized_groups = defaultdict(list)


for original_key, node_id in node_to_id.items():

    normalized_key = (
        str(original_key)
        .strip()
        .lower()
    )

    normalized_groups[normalized_key].append(
        (
            str(original_key).strip(),
            node_id
        )
    )


# ------------------------------------------------------------
# Resolve node id for every unlabeled user
# ------------------------------------------------------------

resolved_node_ids = []
resolution_types = []


for raw_key in true_unlabeled["user_key"].astype(str):

    raw_key = raw_key.strip()


    # 1. Exact match first
    if raw_key in node_to_id:

        resolved_node_ids.append(
            node_to_id[raw_key]
        )

        resolution_types.append(
            "exact"
        )

        continue


    # 2. Normalized fallback
    normalized_key = raw_key.lower()

    candidates = normalized_groups.get(
        normalized_key,
        []
    )


    if len(candidates) == 1:

        resolved_node_ids.append(
            candidates[0][1]
        )

        resolution_types.append(
            "normalized_unique"
        )


    elif len(candidates) > 1:

        resolved_node_ids.append(
            None
        )

        resolution_types.append(
            "ambiguous"
        )


    else:

        resolved_node_ids.append(
            None
        )

        resolution_types.append(
            "missing"
        )


# ------------------------------------------------------------
# Convert to arrays
# ------------------------------------------------------------

resolved_node_ids = np.asarray(
    resolved_node_ids,
    dtype=object
)

resolution_types = np.asarray(
    resolution_types
)


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print(
    "Total unlabeled users:",
    len(true_unlabeled)
)

print(
    "Exact matches:",
    np.sum(
        resolution_types == "exact"
    )
)

print(
    "Normalized unique matches:",
    np.sum(
        resolution_types == "normalized_unique"
    )
)

print(
    "Ambiguous matches:",
    np.sum(
        resolution_types == "ambiguous"
    )
)

print(
    "Missing graph users:",
    np.sum(
        resolution_types == "missing"
    )
)


resolved_mask = np.isin(
    resolution_types,
    [
        "exact",
        "normalized_unique"
    ]
)


print(
    "Total resolved:",
    resolved_mask.sum()
)

print(
    "Resolved coverage:",
    f"{resolved_mask.mean() * 100:.2f}%"
)

SAFE GRAPH NODE RESOLUTION
Total unlabeled users: 18331
Exact matches: 5560
Normalized unique matches: 6837
Ambiguous matches: 0
Missing graph users: 5934
Total resolved: 12397
Resolved coverage: 67.63%


In [18]:
# ============================================================
# BUILD UNLABELED GRAPH FEATURES
# ============================================================

import numpy as np


print("=" * 90)
print("BUILD UNLABELED GRAPH FEATURES")
print("=" * 90)


# ------------------------------------------------------------
# Initialize graph feature matrix
# ------------------------------------------------------------

graph_features_unlabeled = np.zeros(
    (
        len(true_unlabeled),
        degree_features.shape[1]
    ),
    dtype=np.float32
)


graph_available_mask = np.zeros(
    len(true_unlabeled),
    dtype=bool
)


# ------------------------------------------------------------
# Fill graph features using resolved node ids
# ------------------------------------------------------------

for i, node_id in enumerate(resolved_node_ids):

    if node_id is not None:

        node_id = int(node_id)

        graph_features_unlabeled[i] = (
            degree_features[node_id]
        )

        graph_available_mask[i] = True


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print(
    "Graph feature shape:",
    graph_features_unlabeled.shape
)

print(
    "Graph feature dtype:",
    graph_features_unlabeled.dtype
)

print(
    "Available graph users:",
    graph_available_mask.sum()
)

print(
    "Missing graph users:",
    (~graph_available_mask).sum()
)

print(
    "Graph coverage:",
    f"{graph_available_mask.mean() * 100:.2f}%"
)


print(
    "\nNaN values:",
    np.isnan(
        graph_features_unlabeled
    ).sum()
)

print(
    "Infinite values:",
    np.isinf(
        graph_features_unlabeled
    ).sum()
)


# ------------------------------------------------------------
# Alignment checks
# ------------------------------------------------------------

assert (
    graph_features_unlabeled.shape[0]
    == len(true_unlabeled)
)

assert (
    graph_features_unlabeled.shape[1]
    == 3
)

assert (
    graph_available_mask.sum()
    == resolved_mask.sum()
)


print(
    "\nGraph feature alignment: OK"
)

BUILD UNLABELED GRAPH FEATURES
Graph feature shape: (18331, 3)
Graph feature dtype: float32
Available graph users: 12397
Missing graph users: 5934
Graph coverage: 67.63%

NaN values: 0
Infinite values: 0

Graph feature alignment: OK
